# Clase 2 - De Notebook a Sitio Web con Regresion Logistica

Objetivo: entrenar un modelo en notebook y prepararlo para ser consumido por un backend web.

In [50]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1) Dataset sintetico (caso churn)

In [ ]:
rng = np.random.default_rng(42)
n = 1500

df = pd.DataFrame({
    'age': rng.integers(18, 75, size=n),
    'income': rng.normal(35000, 12000, size=n).clip(8000, 120000),
    'tenure_months': rng.integers(1, 72, size=n),
    'support_tickets': rng.poisson(2.0, size=n),
    'is_premium': rng.integers(0, 2, size=n),
})

logit = (
    -2.0
    - 0.015 * df['tenure_months']
    + 0.22 * df['support_tickets']
    - 0.000018 * df['income']
    - 0.55 * df['is_premium']
    + 0.008 * (55 - df['age'])
)
prob = 1 / (1 + np.exp(-logit))
df['churn'] = rng.binomial(1, prob)

df.head()

,age,income,tenure_months,support_tickets,is_premium,churn
0,23,33845.852757,43,3,0,0
1,62,48536.226288,45,1,1,0
2,55,8000.000000,66,3,0,0
3,43,17040.335721,44,1,1,0
4,42,23925.362697,32,1,1,0


## 2) Entrenamiento con Pipeline

In [55]:
features = ['age', 'income', 'tenure_months', 'support_tickets', 'is_premium']
target = 'churn'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ['age', 'income', 'tenure_months', 'support_tickets']
binary_features = ['is_premium']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('bin', 'passthrough', binary_features),
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [ ]:
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, pred, zero_division=0))

Accuracy: 0.9367
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       281
           1       0.00      0.00      0.00        19

    accuracy                           0.94       300
   macro avg       0.47      0.50      0.48       300
weighted avg       0.88      0.94      0.91       300



/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 3) Guardar artefactos para backend

In [57]:
project_root = Path('proyecto_web_logistica')
models_dir = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, models_dir / 'logistic_pipeline.joblib')

metrics = {
    'accuracy': round(float(acc), 4),
    'features': features,
    'model_type': 'LogisticRegression'
}
with open(models_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Modelo y metricas guardados en proyecto_web_logistica/models')

Modelo y metricas guardados en proyecto_web_logistica/models


## 4) Prueba local de inferencia

In [58]:
sample = pd.DataFrame([{
    'age': 30,
    'income': 25000,
    'tenure_months': 6,
    'support_tickets': 4,
    'is_premium': 0
}])

proba = model.predict_proba(sample)[0, 1]
pred = int(proba >= 0.5)
print({'churn_probability': round(float(proba), 4), 'prediction': pred})

{'churn_probability': 0.1998, 'prediction': 0}


## 5) Segundo ejemplo: dataset real de churn (Internet/Telco)

En esta seccion mantenemos el ejemplo sintetico anterior y agregamos un caso con datos reales de clientes de internet.

Dataset: Telco Customer Churn (IBM sample).

Objetivo:
- Cargar un dataset real.
- Entrenar Regresion Logistica.
- Mostrar metricas clave.
- Entender que significa cada metrica en negocio.

In [51]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Carga local primero para evitar bloqueos por red en clase.
fallback_path = Path("/Users/alder.lopez/Documents/ClasesTec/TC3009C.602/CodigoClases/Clase02/proyecto_web_logistica/data/telco_churn.csv")
if fallback_path.exists():
    df_telco = pd.read_csv(fallback_path)
    print("Dataset cargado desde archivo local")
else:
    ds = fetch_openml(name="Telco-Customer-Churn", version=1, as_frame=True)
    df_telco = ds.frame.copy()
    print("Dataset cargado desde OpenML (internet)")

print(df_telco.shape)
df_telco.head()

Dataset cargado desde archivo local
(7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,'No phone service',DSL,No,Yes,No,No,No,No,Month-to-month,Yes,'Electronic check',29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,'One year',No,'Mailed check',56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,'Mailed check',53.85,108.15,Yes
3,Male,0,No,No,45,No,'No phone service',DSL,Yes,No,Yes,Yes,No,No,'One year',No,'Bank transfer (automatic)',42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Electronic check',70.70,151.65,Yes


In [52]:
# Limpieza minima
# El nombre de columnas puede venir en mayuscula/minuscula segun la fuente.
if "TotalCharges" in df_telco.columns:
    df_telco["TotalCharges"] = pd.to_numeric(df_telco["TotalCharges"], errors="coerce")
elif "totalcharges" in df_telco.columns:
    df_telco["totalcharges"] = pd.to_numeric(df_telco["totalcharges"], errors="coerce")

df_telco = df_telco.dropna().copy()

churn_col = "Churn" if "Churn" in df_telco.columns else "churn"
id_col = "customerID" if "customerID" in df_telco.columns else ("customerid" if "customerid" in df_telco.columns else None)

y_telco = (df_telco[churn_col].astype(str).str.lower() == "yes").astype(int)
X_telco = df_telco.drop(columns=[c for c in [id_col, churn_col] if c is not None])

# One-hot para categoricas y escalado para numericas
categorical_cols = X_telco.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_cols = X_telco.select_dtypes(include=["int64", "float64"]).columns.tolist()

X_telco_encoded = pd.get_dummies(X_telco, columns=categorical_cols, drop_first=True)

numeric_after = [c for c in numeric_cols if c in X_telco_encoded.columns]
X_telco_encoded[numeric_after] = StandardScaler().fit_transform(X_telco_encoded[numeric_after])

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_telco_encoded,
    y_telco,
    test_size=0.2,
    random_state=42,
    stratify=y_telco,
)

model_telco = LogisticRegression(max_iter=3000, random_state=42, class_weight="balanced")
model_telco.fit(X_train_t, y_train_t)

pred_t = model_telco.predict(X_test_t)
proba_t = model_telco.predict_proba(X_test_t)[:, 1]

acc_t = accuracy_score(y_test_t, pred_t)
prec_t = precision_score(y_test_t, pred_t)
rec_t = recall_score(y_test_t, pred_t)
f1_t = f1_score(y_test_t, pred_t)
auc_t = roc_auc_score(y_test_t, proba_t)

metrics_telco = pd.DataFrame(
    {
        "metric": ["accuracy", "precision", "recall", "f1", "roc_auc"],
        "value": [acc_t, prec_t, rec_t, f1_t, auc_t],
    }
)
metrics_telco

,metric,value
0,accuracy,0.727079
1,precision,0.491803
2,recall,0.802139
3,f1,0.609756
4,roc_auc,0.835450


In [53]:
cm = confusion_matrix(y_test_t, pred_t)
cm_df = pd.DataFrame(
    cm,
    index=["Real: No Churn", "Real: Churn"],
    columns=["Pred: No Churn", "Pred: Churn"],
)

print("Matriz de confusion")
display(cm_df)

print("\nReporte por clase")
print(classification_report(y_test_t, pred_t, target_names=["No Churn", "Churn"]))

Matriz de confusion


,Pred: No Churn,Pred: Churn
Real: No Churn,723,310
Real: Churn,74,300



Reporte por clase
              precision    recall  f1-score   support

    No Churn       0.91      0.70      0.79      1033
       Churn       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.80      0.73      0.74      1407



## 6) Como interpretar las metricas (explicacion para clase)

- **Accuracy**: porcentaje total de aciertos.
  - Util cuando las clases estan balanceadas.
  - Puede enganarte si hay pocos churners.

- **Precision (clase Churn)**: de los clientes que el modelo marca como churn, cuantos realmente se van.
  - Alta precision = menos falsas alarmas.

- **Recall (clase Churn)**: de todos los clientes que si se van, cuantos detecta el modelo.
  - Alto recall = se escapan menos churners reales.

- **F1-score**: balance entre precision y recall.
  - Sirve cuando quieres equilibrio entre no molestar clientes y no perder churners.

- **ROC-AUC**: capacidad del modelo para separar churn vs no churn a distintos umbrales.
  - Cerca de 1.0 es mejor, cerca de 0.5 es casi aleatorio.

- **Matriz de confusion**:
  - Verdaderos negativos: no churn bien clasificado.
  - Verdaderos positivos: churn bien clasificado.
  - Falsos positivos: cliente estable marcado como churn.
  - Falsos negativos: churn real que el modelo no detecto.

### Pregunta de negocio sugerida

Si tu objetivo es **retener clientes que se van**, normalmente priorizas **recall** en churn y despues ajustas el umbral para controlar precision.